# 기존 뉴스 URL 데이터 추출

`source_data/*.csv`의 모든 URL에서 제목, 주제, 요약, 본문을 추출해 `model/output` 아래에 카테고리별 CSV로 저장합니다. 임베딩 코드는 포함하지 않습니다.

요약은 언론사가 제공한 JSON-LD/OpenGraph 설명을 우선 사용하고, 없으면 본문의 첫 완결 문장 2~3개를 사용합니다. 생성형 요약이 아니므로 원문에 없는 내용을 만들지 않습니다. `summary_source`에 추출 출처를 남깁니다. 주제는 기사 섹션, 키워드, 제목 순으로 선택합니다.

In [ ]:
%pip install -q pandas requests beautifulsoup4 lxml trafilatura tqdm

In [ ]:
from __future__ import annotations
import hashlib, json, re, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urlsplit
import pandas as pd
import requests, trafilatura
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.casefold() == 'model' else cwd
SOURCE_DIR = ROOT / 'source_data'
CACHE_DIR = ROOT / 'model' / 'article_cache'
OUTPUT_DIR = ROOT / 'model' / 'output'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CRAWL_LIMIT: int | None = None  # 모든 고유 URL 수집
REQUEST_DELAY_SECONDS = 0.25
REQUEST_TIMEOUT_SECONDS = 20
MAX_BODY_CHARS = 50_000
SUMMARY_MAX_CHARS = 500
RETRY_FAILED_CACHE = True
EXTRACTION_VERSION = 2  # 광고 정제 규칙 버전
print('프로젝트:', ROOT)
print('카테고리별 출력 폴더:', OUTPUT_DIR)

## 1. 원본 CSV 목록과 개별 로더

In [ ]:
CATEGORY_MAP = {
    'Innodep': '이노뎁 관련 기사', 'Security': '보안 관련 기사',
    'Industry Trends': '업계 동향 기사', 'Government/Public': '정부/공공 기사',
    'Venture/Finance': '벤처/금융 기사', 'Production/Wages': '생산/임금 기사',
}
files = sorted(SOURCE_DIR.glob('*.csv'))
if not files: raise FileNotFoundError(SOURCE_DIR)

def load_source_csv(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    legacy = pd.read_csv(path, dtype=str)
    missing = {'date', 'category', 'article_link'} - set(legacy.columns)
    if missing: raise ValueError(f'{path.name}: 필수 컬럼 누락 {missing}')
    legacy['source_file'] = path.name
    legacy['digest_date'] = pd.to_datetime(legacy['date'], errors='raise').dt.date.astype(str)
    legacy['legacy_category'] = legacy['category'].map(CATEGORY_MAP)
    if legacy['legacy_category'].isna().any(): raise ValueError(f'{path.name}: 알 수 없는 카테고리')
    legacy['article_link'] = legacy['article_link'].str.strip()
    legacy['article_id'] = legacy['article_link'].map(lambda x: hashlib.sha256(x.encode()).hexdigest()[:24])
    articles = legacy.drop_duplicates('article_link')[['article_id','article_link']].reset_index(drop=True)
    return legacy, articles

file_summary = []
for source_path in files:
    source_legacy, source_articles = load_source_csv(source_path)
    file_summary.append({'file':source_path.name,'rows':len(source_legacy),'unique_urls':len(source_articles)})
display(pd.DataFrame(file_summary))

## 2. 제목·주제·요약·본문 추출 함수

In [ ]:
def clean(value: Any) -> str:
    if value is None: return ''
    return re.sub(r'\s+', ' ', BeautifulSoup(str(value), 'html.parser').get_text(' ', strip=True)).strip()

BOILERPLATE_PATTERNS = [
    r'^(ADVERTISEMENT|Sponsored|광고 닫기)$',
    r'^(관련 ?기사|추천 ?기사|많이 본 뉴스|실시간 인기기사)( 더보기)?$',
    r'^(기자에게 제보하기|기사제보|구독하기|로그인|SNS 공유)$',
    r'^무단전재\s*및\s*재배포\s*금지',
    r'^저작권자\s*[©ⓒ]',
    r'^[가-힣]{2,5}\s*기자\s*([|·,:-]?\s*[\w.+-]+@[\w.-]+\.[A-Za-z]{2,})?$',
    r'^[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}$',
]
REMOVE_SELECTORS = [
    'script','style','nav','footer','aside','iframe','form','noscript',
    '[class*="advert"]','[id*="advert"]','[class*="advertisement"]',
    '[id*="advertisement"]','[class*="banner"]','[id*="banner"]',
    '[class*="related-news"]','[class*="recommend"]','[class*="comment"]',
    '[class*="social-share"]','[class*="subscription"]','[class*="newsletter"]',
]

def remove_noise_nodes(soup: BeautifulSoup) -> BeautifulSoup:
    for selector in REMOVE_SELECTORS:
        for node in soup.select(selector): node.decompose()
    return soup

def clean_article_body(value: Any) -> str:
    if value is None: return ''
    text = BeautifulSoup(str(value), 'html.parser').get_text('\n', strip=True)
    text = text.replace('\xa0',' ').replace('\r','\n')
    result, seen = [], set()
    for raw_line in text.splitlines():
        line = re.sub(r'\s+',' ',raw_line).strip()
        if not line or any(re.search(p,line,re.IGNORECASE) for p in BOILERPLATE_PATTERNS): continue
        if len(line) <= 5: continue
        key = re.sub(r'\s+','',line).casefold()
        if key in seen: continue
        seen.add(key); result.append(line)
    return '\n'.join(result)

def walk_json(value: Any):
    if isinstance(value, dict):
        yield value
        for child in value.values(): yield from walk_json(child)
    elif isinstance(value, list):
        for child in value: yield from walk_json(child)

def keyword_text(value: Any) -> str:
    items = value if isinstance(value, list) else re.split(r'[,|]', clean(value))
    result = []
    for item in items:
        item = clean(item)
        if item and item.casefold() not in [x.casefold() for x in result]: result.append(item)
    return ', '.join(result[:10])

def json_ld_article(soup: BeautifulSoup) -> dict[str, str]:
    best = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try: payload = json.loads(tag.string or tag.get_text())
        except Exception: continue
        for node in walk_json(payload):
            kinds = node.get('@type', [])
            kinds = kinds if isinstance(kinds, list) else [kinds]
            if not any(str(x).casefold() in {'newsarticle','article','reportagenewsarticle'} for x in kinds): continue
            pub = node.get('publisher', '')
            if isinstance(pub, dict): pub = pub.get('name', '')
            candidate = {
                'title': clean(node.get('headline') or node.get('name')),
                'description': clean(node.get('description')),
                'body': clean(node.get('articleBody')),
                'section': clean(node.get('articleSection')),
                'keywords': keyword_text(node.get('keywords')),
                'published_at': clean(node.get('datePublished')), 'publisher': clean(pub),
            }
            if len(candidate['body']) + 10*bool(candidate['title']) > len(best.get('body','')) + 10*bool(best.get('title')): best = candidate
    return best

def meta(soup: BeautifulSoup, *selectors: str) -> str:
    for selector in selectors:
        tag = soup.select_one(selector)
        if tag and clean(tag.get('content')): return clean(tag.get('content'))
    return ''

def html_body(soup: BeautifulSoup) -> str:
    remove_noise_nodes(soup)
    selectors = ['article','#articleBody','#article-body','#newsEndContents','.article_body','.article-body','.news_body','.view_body','.article_txt']
    candidates = [clean_article_body(n.get_text('\n',strip=True)) for s in selectors for n in soup.select(s)]
    candidates = [x for x in candidates if x]
    if candidates: return max(candidates, key=len)
    return clean_article_body('\n'.join(p.get_text(' ',strip=True) for p in soup.select('p')))

def lead_summary(body: str, max_sentences: int = 3) -> str:
    sentences = re.split(r'(?<=[.!?。！？])\s+', clean(body))
    selected = []
    for sentence in sentences:
        if len(sentence) < 20: continue
        if len(' '.join(selected + [sentence])) > SUMMARY_MAX_CHARS:
            if not selected: selected = [sentence[:SUMMARY_MAX_CHARS].rstrip() + '…']
            break
        selected.append(sentence)
        if len(selected) == max_sentences: break
    return ' '.join(selected)

def select_summary(tf_desc: str, ld_desc: str, meta_desc: str, body: str):
    for source, value in [('trafilatura_description',tf_desc),('json_ld_description',ld_desc),('meta_description',meta_desc)]:
        value = clean_article_body(value)
        if len(value) >= 30: return value[:SUMMARY_MAX_CHARS], source
    value = lead_summary(body)
    return value, ('lead_sentences' if value else 'missing')

## 3. 개별 CSV용 URL 수집 함수

기사별 JSON 캐시를 사용하므로 중단 후 다시 실행할 수 있습니다.

In [ ]:
retry = Retry(total=3, backoff_factor=.8, status_forcelist=(429,500,502,503,504), allowed_methods=frozenset({'GET'}))
session = requests.Session()
session.mount('http://', HTTPAdapter(max_retries=retry)); session.mount('https://', HTTPAdapter(max_retries=retry))
session.headers.update({'User-Agent':'Mozilla/5.0 (compatible; NewsDigestResearch/1.0)','Accept-Language':'ko-KR,ko;q=0.9'})

def extract_url(url: str, article_id: str) -> dict[str, Any]:
    response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS); response.raise_for_status()
    response.encoding = response.apparent_encoding or response.encoding
    html = response.text; soup = BeautifulSoup(html, 'lxml'); ld = json_ld_article(soup)
    content_soup = remove_noise_nodes(BeautifulSoup(html, 'lxml'))
    cleaned_html = str(content_soup)
    tf = {}
    try:
        raw = trafilatura.extract(cleaned_html,url=response.url,output_format='json',with_metadata=True,include_comments=False,include_tables=False,include_links=False,favor_precision=True,deduplicate=True)
        if raw: tf = json.loads(raw)
    except Exception: pass
    title = clean(tf.get('title') or ld.get('title') or meta(soup,'meta[property="og:title"]') or (soup.title.get_text() if soup.title else ''))
    body = clean_article_body(tf.get('text') or ld.get('body') or html_body(content_soup))[:MAX_BODY_CHARS]
    section = clean(ld.get('section') or meta(soup,'meta[property="article:section"]'))
    keywords = keyword_text(ld.get('keywords') or meta(soup,'meta[name="keywords"]','meta[property="article:tag"]'))
    topic, topic_source = (section,'article_section') if section else ((keywords,'keywords') if keywords else (title,'title_fallback'))
    summary, summary_source = select_summary(tf.get('description',''), ld.get('description',''), meta(soup,'meta[property="og:description"]','meta[name="description"]'), body)
    publisher = clean(ld.get('publisher') or meta(soup,'meta[property="og:site_name"]') or urlsplit(response.url).netloc.removeprefix('www.'))
    published = clean(tf.get('date') or ld.get('published_at') or meta(soup,'meta[property="article:published_time"]'))
    ok = bool(title and len(body) >= 200)
    return {'article_id':article_id,'extraction_version':EXTRACTION_VERSION,'final_url':response.url,'title':title,'topic':topic,'topic_source':topic_source,'summary':summary,'summary_source':summary_source,'body':body,'published_at':published,'publisher':publisher,'http_status':response.status_code,'fetch_status':'success' if ok else 'incomplete','error':'' if ok else '제목 누락 또는 정제 본문 200자 미만','fetched_at_utc':datetime.now(timezone.utc).isoformat(),'ok':ok}

def read_cache(path: Path):
    try: return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None
    except Exception: return None
def write_cache(path: Path, data: dict):
    temp = path.with_suffix('.tmp'); temp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8'); temp.replace(path)

def crawl_source(articles: pd.DataFrame, cache_dir: Path, description: str) -> dict[str,int]:
    cache_dir.mkdir(parents=True,exist_ok=True)
    targets = articles if CRAWL_LIMIT is None else articles.head(CRAWL_LIMIT)
    stats = {'cached':0,'success':0,'failed_or_incomplete':0}
    for row in tqdm(targets.itertuples(index=False),total=len(targets),desc=description):
        path = cache_dir / f'{row.article_id}.json'; cached = read_cache(path)
        cache_is_current = cached and cached.get('extraction_version') == EXTRACTION_VERSION
        if cache_is_current and (cached.get('ok') or not RETRY_FAILED_CACHE): stats['cached'] += 1; continue
        try: data = extract_url(row.article_link,row.article_id)
        except Exception as exc:
            data = {'article_id':row.article_id,'extraction_version':EXTRACTION_VERSION,'final_url':'','title':'','topic':'','topic_source':'missing','summary':'','summary_source':'missing','body':'','published_at':'','publisher':'','http_status':getattr(getattr(exc,'response',None),'status_code',None),'fetch_status':'failed','error':f'{type(exc).__name__}: {exc}','fetched_at_utc':datetime.now(timezone.utc).isoformat(),'ok':False}
        stats['success' if data['ok'] else 'failed_or_incomplete'] += 1
        write_cache(path,data); time.sleep(REQUEST_DELAY_SECONDS)
    return stats

## 4. 원본 CSV별 독립 수집 및 개별 결과 저장

In [ ]:
columns = ['digest_date','legacy_category','article_link','article_id','extraction_version','title','topic','topic_source','summary','summary_source','body','published_at','publisher','final_url','http_status','fetch_status','error','fetched_at_utc','source_file']
def process_source(filename: str) -> pd.DataFrame:
    source_path = SOURCE_DIR / filename
    if not source_path.exists(): raise FileNotFoundError(source_path)
    legacy, articles = load_source_csv(source_path)
    source_cache_dir = CACHE_DIR / source_path.stem
    stats = crawl_source(articles,source_cache_dir,source_path.stem)
    records = [data for row in articles.itertuples(index=False) if (data := read_cache(source_cache_dir / f'{row.article_id}.json'))]
    extracted = pd.DataFrame(records) if records else pd.DataFrame(columns=['article_id'])
    output = legacy.merge(extracted,on='article_id',how='left',validate='many_to_one')
    output['fetch_status'] = output.get('fetch_status',pd.Series(index=output.index,dtype=str)).fillna('not_fetched')
    for column in columns:
        if column not in output: output[column] = ''
    output = output[columns].sort_values(['digest_date','article_id']).reset_index(drop=True)
    output_path = OUTPUT_DIR / f'{source_path.stem}_enriched.csv'
    output.to_csv(output_path,index=False,encoding='utf-8-sig')
    result = {'input':source_path.name,'output':output_path.name,'rows':len(output),'unique_urls':len(articles),**stats}
    display(pd.DataFrame([result]))
    display(output['fetch_status'].value_counts(dropna=False).to_frame('rows'))
    print(f'{source_path.name} 완료 -> {output_path.name}')
    return output

## 5-1. 이노뎁 기사 수집

In [ ]:
category_output = process_source('news_innodep.csv')

## 5-2. 보안 기사 수집

In [ ]:
category_output = process_source('news_security.csv')

## 5-3. 업계 동향 기사 수집

In [ ]:
category_output = process_source('news_industry_trends.csv')

## 5-4. 정부·공공 기사 수집

In [ ]:
category_output = process_source('news_government_public.csv')

## 5-5. 벤처·금융 기사 수집

In [ ]:
category_output = process_source('news_venture_finance.csv')

## 5-6. 생산·임금 기사 수집

In [ ]:
category_output = process_source('news_production_wages.csv')